# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

#выйдет {'A': 2, 'B': 1} потому что перебирает буквы сколько раз встречается буквы 

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# [0, 1, 1]. Потому что функция изменяет тот же самый список a, а не создаёт новый.

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))
# выйдет реузльтат два потому что количество элементов два 
# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)
#Результат: {'ab': 2, 'abc': 3}. потому что в словаре ключи не могут повторяться 
# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

# вот тут выйдет число 5 
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:
# База
#1 
def to_num(s):
    return float(str(s).replace(" ", "").replace(",", "."))

def clean_city(s):
    s = s.strip().lower()

    cities = {
        "almaty": "Алматы",
        "astana": "Астана",
        "shymkent": "Шымкент",
        "алматы": "Алматы",
        "астана": "Астана",
        "шымкент": "Шымкент",
        "караганда": "Караганда",
        "актобе": "Актобе",
        "атырау": "Атырау"
    }

    return cities.get(s, s.title())


In [ ]:
#2 
counts = {}

for order in orders:
    category = order["category"]
    counts[category] = counts.get(category, 0) + 1

result = sorted(counts.items(), key=lambda x: x[1], reverse=True)

for category, count in result:
    print(category, count)


In [ ]:
for category in counts:
    total = 0
    count = 0

    for order in orders:
        if order["category"] == category and order["status"] == "оплачен":
            total += to_num(order["amount"])
            count += 1

    average = total / count if count > 0 else 0

    print(f"{category} | {total:.0f} ₸ | {average:.0f} ₸")

In [ ]:
months = {}

for order in orders:
    month = order["order_date"][:7]
    months[month] = months.get(month, 0) + 1

for month, count in sorted(months.items()):
    print(f"{month} | {count} заказов") 

## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# Стандарт
revenue = {}

for order in orders:
    if order["status"] == "оплачен":
        month = order["order_date"][:7]
        amount = to_num(order["amount"])

        revenue[month] = revenue.get(month, 0) + amount

september = revenue["2026-09"]
october = revenue["2026-10"]

change = (october - september) / september * 100

print("Сентябрь:", september)
print("Октябрь:", october)
print(f"Изменение: {change:.1f}%")

In [ ]:
total_city = {}
cancel_city = {}

for order in orders:
    city = clean_city(order["city"])

    total_city[city] = total_city.get(city, 0) + 1

    if order["status"] == "отменён":
        cancel_city[city] = cancel_city.get(city, 0) + 1

for city in total_city:
    total = total_city[city]
    cancelled = cancel_city.get(city, 0)

    percent = cancelled / total * 100

    print(f"{city} | {percent:.1f}%")

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж